In [3]:
!pip install -q \
    langgraph \
    chromadb \
    google-genai \
    groq
print("Libraries installed successfully!")

Libraries installed successfully!


In [4]:
from typing import TypedDict

from langgraph.graph import (
    StateGraph,
    START,
    END
)

try:
    print("LangGraph imports successful!")

except Exception as e:
    print("Import Error:")
    print(e)

LangGraph imports successful!


In [5]:

try:

    class GraphState(TypedDict):

        question: str

        query_embedding: list

        retrieved_chunks: list

        answer: str

    print("GraphState created successfully!")

except Exception as e:

    print("State Error:")
    print(e)

GraphState created successfully!


In [6]:
from google import genai
from google.colab import userdata

try:
    gemini_client = genai.Client(
        api_key=userdata.get("GEMINI_KEY")
    )

    print("Gemini connected successfully!")

except Exception as e:
    print("Gemini Connection Error:")
    print(e)

Gemini connected successfully!


In [10]:
import pickle
import chromadb

try:
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    if collection.count() == 0:
        collection.add(
            ids=[f"chunk_{i}" for i in range(len(chunks))],
            documents=chunks,
            embeddings=chunk_embeddings
        )

    print("ChromaDB connected successfully!")
    print("Records:", collection.count())

except Exception as e:
    print("ChromaDB Error:")
    print(e)

ChromaDB connected successfully!
Records: 247


In [11]:
from groq import Groq
from google.colab import userdata

try:
    groq_client = Groq(
        api_key=userdata.get("GROQ_KEY")
    )

    print("Groq connected successfully!")

except Exception as e:
    print("Groq Connection Error:")
    print(e)

Groq connected successfully!


In [12]:
try:

    def embed_question(state):

        question = state["question"]

        response = gemini_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        state["query_embedding"] = (
            response.embeddings[0].values
        )

        print("Question embedding generated!")

        return state

    print("Embed Question Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Embed Question Node created!


In [13]:

try:

    test_state = {
        "question": "What is machine learning?"
    }

    result = embed_question(test_state)

    print("Embedding Dimension:")
    print(len(result["query_embedding"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Embedding Dimension:
3072


In [14]:
try:

    def retrieve_chunks(state):

        results = collection.query(
            query_embeddings=[
                state["query_embedding"]
            ],
            n_results=5
        )

        state["retrieved_chunks"] = (
            results["documents"][0]
        )

        print("Top 5 chunks retrieved!")

        return state

    print("Retrieve Chunks Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Retrieve Chunks Node created!


In [15]:
try:

    state = {
        "question": "What is Machine Learning?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    print("\nRetrieved Chunks:")
    print(len(state["retrieved_chunks"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!

Retrieved Chunks:
5


In [16]:
try:

    for i, chunk in enumerate(
        state["retrieved_chunks"],
        start=1
    ):

        print("=" * 60)
        print(f"Chunk {i}")
        print(chunk[:500])
        print()

except Exception as e:

    print(e)

Chunk 1
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations o

Chunk 2
Models
A machine learning model is a type of mathematical model that, once "trained" on a given dataset, can be used to make predictions or classifications on new data. During training, a learning algorithm iteratively adjusts the model's internal parameters to minimise errors in its predictions. By extension, the term "model" can refer to several levels of specificity, from a general class of models and their associated learning algorithms to a fully trained model with all its

In [17]:
try:

    def generate_answer(state):

        context = "\n\n".join(
            state["retrieved_chunks"]
        )

        prompt = f"""
You are a helping assistant.

Answer only using the provided context.

If the answer is not available in the context,
respond with:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{state["question"]}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        state["answer"] = (
            response.choices[0].message.content
        )

        print("Answer generated successfully!")

        return state

    print("Generate Answer Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Generate Answer Node created!


In [18]:
try:

    state = {
        "question": "What is the difference between AI and machine learning?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    state = generate_answer(state)

    print("\nFinal Answer:\n")
    print(state["answer"])

except Exception as e:

    print("Pipeline Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Machine learning (ML) grew out of the quest for artificial intelligence (AI), but over time, the field of ML shifted its focus away from symbolic approaches inherited from AI and toward methods and models borrowed from statistics, fuzzy logic, and probability theory. In the early days, AI and ML were closely related, but an increasing emphasis on the logical, knowledge-based approach in AI caused a rift between the two fields. While AI focused on expert systems and symbolic/knowledge-based learning, ML focused on statistical and probabilistic approaches, eventually becoming a distinct field.


In [19]:
try:

    graph_builder = StateGraph(GraphState)

    graph_builder.add_node(
        "embed_question",
        embed_question
    )

    graph_builder.add_node(
        "retrieve_chunks",
        retrieve_chunks
    )

    graph_builder.add_node(
        "generate_answer",
        generate_answer
    )

    print("Nodes added successfully!")

except Exception as e:

    print("Graph Creation Error:")
    print(e)

Nodes added successfully!


In [20]:
try:

    graph_builder.add_edge(
        START,
        "embed_question"
    )

    graph_builder.add_edge(
        "embed_question",
        "retrieve_chunks"
    )

    graph_builder.add_edge(
        "retrieve_chunks",
        "generate_answer"
    )

    graph_builder.add_edge(
        "generate_answer",
        END
    )

    print("Edges connected successfully!")

except Exception as e:

    print("Edge Error:")
    print(e)

Edges connected successfully!


In [21]:
try:

    graph = graph_builder.compile()

    print("LangGraph compiled successfully!")

except Exception as e:

    print("Compilation Error:")
    print(e)

LangGraph compiled successfully!


In [23]:
try:

    result = graph.invoke(
        {
            "question":
            "What is the difference between AI and machine learning?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except Exception as e:

    print("Execution Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Machine learning grew out of the quest for artificial intelligence (AI), but the field changed its goal from achieving artificial intelligence to tackling solvable problems of a practical nature. It shifted focus away from the symbolic approaches it had inherited from AI, and toward methods and models borrowed from statistics, fuzzy logic, and probability theory. An increasing emphasis on the logical, knowledge-based approach caused a rift between AI and machine learning, with AI focusing on expert systems and machine learning focusing on statistical and probabilistic approaches.


In [25]:

result = graph.invoke(
    {
        "question":
        "What is deep learning?"
    }


)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
Deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representation. It involves utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning, with the adjective "deep" referring to the use of multiple layers in the network.


In [26]:
result = graph.invoke(
    {
        "question":
        "What is natural language processing?"
    }
)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
Natural language processing (NLP) is the processing of natural language information by a computer. It is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.
